In [4]:
%load_ext autoreload
%autoreload 2
from seek_code import SEEK
from data_handler import EleHandler
from classifier import Classifier
from backbone import Backbone
from concept_head import ConceptHead
from projector import Projector

from torch.utils.data import DataLoader, Subset
from torch.utils.data.sampler import RandomSampler

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Data

In [ ]:
dataset = EleHandler(EFA_IDI_only=True)

train_indices, val_indices, test_indices = dataset.split_along_encounters([0.7,0.15,0.15])

train_loader, val_loader, test_loader = (DataLoader(dataset, batch_size=128, sampler=RandomSampler(indices), num_workers=4) for indices in [train_indices, val_indices, test_indices])

# Check the values of the labels
max_ele_label = 0
for batch in train_loader:
    max_ele_label = max(torch.max(batch[2]), max_ele_label)

print('Number of elephants:', max_ele_label.item() + 1)

code = 'messing_around'

Train indices: 3904, Validation indices: 729, Test indices: 715
Number of elephants: 999


### Training stage 1 : Concept Head
- loading a pretrained backbone, freeze it
- Train the concept head to learn the SEEK codes
- Test the concept head


In [ ]:

ba = Backbone(pretraining = "savannah_elephants")
ba.freeze()

ch = ConceptHead(print_every=1, experiment_code=code)
#ch.train(train_loader, val_loader, ba, num_epochs=30)
ch.test(test_loader, ba)

ch.freeze()

### Training stage 2 : Cassifier

- With a frozen backbone, train a classifier to learn the elephant id from the backbone original embeddings
- Freeze the classifier|


In [8]:

cl = Classifier( num_classes=(max_ele_label + 1), experiment_code=code)
#cl.train(train_loader, val_loader, backbone = ba, num_epochs=3)
cl.test(test_loader, backbone = ba) 

# Freeze the classifier layer
cl.freeze()

found weights in /data/vision/beery/scratch/antoine/CBM_reid/weights/classifier_w.pt ....loading them


/data/vision/beery/scratch/antoine/CBM_reid/methods/classifier.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.layer.load_state_dict(torch.load(Path.Path.cwd().par

Classifier - Test Loss: 6.5359, Test Accuracy: 37.60%


### Training stage 3 : Projector

- With a frozen backbone and the frozen classifier, train the projector
- The projector trains to project the concepts to the embedding space 
- The loss is computed such that edited embeddings (original embeddings + projected embeedings) correctly predict the elephant id using the classifier learnt is stage 2

In [ ]:
def no_intervention(predicted_concepts, true_concepts, correction_probability = 0.0):
    return predicted_concepts

In [ ]:
pr = Projector(experiment_code=code) 

#Freeze the concept head layer
ba.freeze()
ch.freeze()
cl.freeze()
pr.unfreeze()

pr.train(train_loader, val_loader, ba,  ch, cl, num_epochs=50, intervention_fn=no_intervention)
pr.test(test_loader, ba, cl, ch)


training projector, reinitialized weights
Concept head parameters require gradients: False Classifier parameters require gradients: False Backbone parameters require gradients: False Projector parameters require gradients: True


Epoch 1/50 - Train Loss: 6.5618, Train Accuracy: 35.23% - Val Loss: 6.5404, Val Accuracy: 36.98%
Epoch 2/50 - Train Loss: 6.5592, Train Accuracy: 35.28% - Val Loss: 6.5321, Val Accuracy: 38.02%
Epoch 3/50 - Train Loss: 6.5530, Train Accuracy: 36.04% - Val Loss: 6.5336, Val Accuracy: 37.89%
Epoch 4/50 - Train Loss: 6.5541, Train Accuracy: 35.76% - Val Loss: 6.5291, Val Accuracy: 38.34%
Epoch 5/50 - Train Loss: 6.5488, Train Accuracy: 36.39% - Val Loss: 6.5251, Val Accuracy: 38.70%
Epoch 6/50 - Train Loss: 6.5481, Train Accuracy: 36.42% - Val Loss: 6.5278, Val Accuracy: 38.26%
Epoch 7/50 - Train Loss: 6.5447, Train Accuracy: 36.59% - Val Loss: 6.5207, Val Accuracy: 38.90%
Epoch 8/50 - Train Loss: 6.5430, Train Accuracy: 36.84% - Val Loss: 6.5216, Val Accuracy: 38.75%
Epoch 9/50 - Train Loss: 6.5412, Train Accuracy: 36.90% - Val Loss: 6.5186, Val Accuracy: 39.24%
Epoch 10/50 - Train Loss: 6.5408, Train Accuracy: 37.05% - Val Loss: 6.5193, Val Accuracy: 39.25%
Epoch 11/50 - Train Loss: 6.5

### Training stage 4 : Projector and Classifier jointly under partial intervention

- With the same architecture, unfreeze the classifier and the projector. Keep the concept head and the backbone frozen
- Train jointly the Projector and Classifier to maximize elephant classification
- When training, correct the concepts with probability 0.5

In [ ]:
print('intervention at level 0.5')

def intervene(predicted_concepts, true_concepts, correction_probability = 0.5):
    for i in range(predicted_concepts.shape[0]):
        if torch.rand(1).item() < correction_probability:
            predicted_concepts[i] = true_concepts[i]
    # To-do correction
    return predicted_concepts

pr.unfreeze()
cl.unfreeze()
ch.freeze()
ba.freeze()

pr.train(train_loader, val_loader,  ch, cl, num_epochs=10, intervention_fn = intervene)
pr.test(test_loader, cl, ch)